# 02 — Amenity Composition

Queries OSM via Overpass API for amenities within each census tract and computes per-tract counts, ratios, and density.

**Data source:** Overpass API (OSM) — fully portable to any city.

**Method:** Single batch Overpass query for all amenity/shop/office/craft/tourism/leisure POIs in the study area, then BallTree matching to assign each POI to its nearest tract and categorize.

**Output columns:** `tract_id`, `amenity_count_total`, `amenity_count_food`, `amenity_count_retail`, `amenity_count_services`, `amenity_count_office`, `amenity_count_hotel`, `amenity_count_entertainment`, `amenity_count_education`, `amenity_count_healthcare`, `amenity_ratio_food`, `amenity_ratio_retail`, `amenity_density`

**Output file:** `csv/02_amenity_composition.csv`

In [ ]:
# ── Papermill parameters ──────────────────────────────
ZONES_CONFIG = "zones.json"
QUERY_RADIUS = 500   # meters around each tract centroid

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import json
import os
import math
import hashlib
from sklearn.neighbors import BallTree

os.makedirs("csv", exist_ok=True)
os.makedirs("cache", exist_ok=True)

df_tracts = pd.read_csv("csv/01_zone_definition.csv", dtype={"tract_id": str})
print(f"Loaded {len(df_tracts)} tracts")

# Bounding box for batch query
BUFFER = 0.015
LAT_MIN = df_tracts["tract_lat"].min() - BUFFER
LAT_MAX = df_tracts["tract_lat"].max() + BUFFER
LON_MIN = df_tracts["tract_lon"].min() - BUFFER
LON_MAX = df_tracts["tract_lon"].max() + BUFFER
BBOX = f"{LAT_MIN},{LON_MIN},{LAT_MAX},{LON_MAX}"

In [ ]:
# ── Overpass API configuration ────────────────────────

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "zone-finding/1.0 (research project)"}

# Tags to query — broad set for zone characterization
QUERY_TAGS = ["amenity", "shop", "office", "craft", "tourism", "leisure"]

# ── Amenity categorization ────────────────────────────
# Maps primary_value → category for zone-level aggregation
AMENITY_CATEGORIES = {
    # food_drink
    "restaurant": "food_drink", "fast_food": "food_drink",
    "cafe": "food_drink", "bar": "food_drink", "pub": "food_drink",
    "food_court": "food_drink", "bakery": "food_drink",
    "ice_cream": "food_drink", "deli": "food_drink",
    "confectionery": "food_drink", "coffee": "food_drink",
    "tea": "food_drink", "beverages": "food_drink",
    "butcher": "food_drink", "greengrocer": "food_drink",
    "pastry": "food_drink", "brewery": "food_drink",
    "grocery": "food_drink", "seafood": "food_drink",
    # retail
    "clothes": "retail", "shoes": "retail", "jewelry": "retail",
    "gift": "retail", "convenience": "retail", "supermarket": "retail",
    "department_store": "retail", "mobile_phone": "retail",
    "electronics": "retail", "books": "retail", "furniture": "retail",
    "hardware": "retail", "cosmetics": "retail", "watches": "retail",
    "bag": "retail", "art": "retail", "antiques": "retail",
    "florist": "retail", "alcohol": "retail", "wine": "retail",
    "stationery": "retail", "toys": "retail", "pet": "retail",
    "variety_store": "retail", "marketplace": "retail",
    "fabric": "retail", "fashion_accessories": "retail",
    "perfumery": "retail", "chemist": "retail", "tobacco": "retail",
    "newsagent": "retail", "kiosk": "retail",
    # office
    "company": "office", "lawyer": "office", "accountant": "office",
    "it": "office", "consulting": "office", "estate_agent": "office",
    "financial_advisor": "office", "architect": "office",
    "tax_advisor": "office", "marketing": "office",
    "property_management": "office", "engineer": "office",
    "advertising_agency": "office", "ngo": "office",
    "coworking": "office", "coworking_space": "office",
    # hotel
    "hotel": "hotel", "hostel": "hotel", "motel": "hotel",
    "guest_house": "hotel",
    # entertainment
    "theatre": "entertainment", "cinema": "entertainment",
    "nightclub": "entertainment", "arts_centre": "entertainment",
    "gallery": "entertainment", "museum": "entertainment",
    "events_venue": "entertainment", "escape_game": "entertainment",
    "fitness_centre": "entertainment", "sports_centre": "entertainment",
    "karaoke": "entertainment", "dance": "entertainment",
    "swimming_pool": "entertainment",
    # education
    "school": "education", "university": "education",
    "college": "education", "kindergarten": "education",
    "library": "education", "language_school": "education",
    "music_school": "education", "driving_school": "education",
    # healthcare
    "pharmacy": "healthcare", "dentist": "healthcare",
    "clinic": "healthcare", "doctors": "healthcare",
    "hospital": "healthcare", "optician": "healthcare",
    "veterinary": "healthcare",
    # services (catch-all for remaining commercial)
    "hairdresser": "services", "beauty": "services",
    "massage": "services", "tattoo": "services",
    "laundry": "services", "dry_cleaning": "services",
    "bank": "services", "atm": "services",
    "bureau_de_change": "services", "travel_agency": "services",
    "photo": "services", "copyshop": "services",
    "tailor": "services", "shoemaker": "services",
}

CATEGORIES = ["food_drink", "retail", "services", "office", "hotel",
              "entertainment", "education", "healthcare"]

print(f"Amenity categories: {len(CATEGORIES)}")
print(f"Mapped values: {len(AMENITY_CATEGORIES)}")

In [ ]:
# ── Overpass query and caching ────────────────────────

def _cache_path(query):
    h = hashlib.sha1(query.encode()).hexdigest()
    return f"cache/{h}.json"


def query_overpass_cached(query, max_retries=3):
    """Query Overpass API with file-based caching."""
    cp = _cache_path(query)
    if os.path.exists(cp):
        with open(cp, encoding="utf-8") as f:
            return json.load(f)
    last_error = None
    for attempt in range(max_retries):
        ep = OVERPASS_ENDPOINTS[attempt % len(OVERPASS_ENDPOINTS)]
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=120)
            r.raise_for_status()
            data = r.json()
            with open(cp, "w", encoding="utf-8") as f:
                json.dump(data, f)
            return data
        except Exception as e:
            last_error = e
            time.sleep(3 + attempt * 2)
    raise RuntimeError(f"Overpass query failed after {max_retries} retries: {last_error}")


print("Query functions ready.")

In [ ]:
# ── Batch query: ALL amenity POIs in study area ───────

tag_filters = "\n ".join(
    [f'node["{tag}"]({BBOX});' for tag in QUERY_TAGS]
    + [f'way["{tag}"]({BBOX});' for tag in QUERY_TAGS]
)
query = f"[out:json][timeout:120];\n(\n {tag_filters}\n);\nout center tags;"

print("Querying all amenity POIs in study area...")
data = query_overpass_cached(query)

# ── Extract and categorize POIs ───────────────────────
EARTH_RADIUS_M = 6371000
TRACT_AREA_KM2 = math.pi * (QUERY_RADIUS / 1000) ** 2
MAX_DIST_M = QUERY_RADIUS

poi_records = []
for el in data.get("elements", []):
    tags = el.get("tags", {})
    lat = el.get("lat") or (el.get("center", {}) or {}).get("lat")
    lon = el.get("lon") or (el.get("center", {}) or {}).get("lon")
    if not (lat and lon):
        continue
    # Find the primary category from our query tags
    cat = None
    for tag_key in QUERY_TAGS:
        if tag_key in tags:
            cat = AMENITY_CATEGORIES.get(tags[tag_key])
            break
    if cat:
        poi_records.append({"lat": float(lat), "lon": float(lon), "category": cat})

print(f"  Found {len(poi_records)} categorized POIs")

# ── BallTree: assign each POI to nearest tract ────────
tract_coords_rad = np.radians(df_tracts[["tract_lat", "tract_lon"]].values)
tract_ids = df_tracts["tract_id"].tolist()

# Per-tract category accumulators
tract_counts = {t: {cat: 0 for cat in CATEGORIES} for t in tract_ids}

if poi_records:
    tract_tree = BallTree(tract_coords_rad, metric="haversine")
    poi_coords = np.radians([[p["lat"], p["lon"]] for p in poi_records])
    distances, indices = tract_tree.query(poi_coords, k=1)

    for j, (dist, idx) in enumerate(zip(distances.flatten(), indices.flatten())):
        if dist * EARTH_RADIUS_M <= MAX_DIST_M:
            tid = tract_ids[idx]
            tract_counts[tid][poi_records[j]["category"]] += 1

# ── Build result DataFrame ────────────────────────────
records = []
for tid in tract_ids:
    counts = tract_counts[tid]
    total = sum(counts.values())

    rec = {"tract_id": tid, "amenity_count_total": total}
    for cat in CATEGORIES:
        rec[f"amenity_count_{cat}"] = counts[cat]

    if total > 0:
        for cat in CATEGORIES:
            rec[f"amenity_ratio_{cat}"] = round(counts[cat] / total, 4)
    else:
        for cat in CATEGORIES:
            rec[f"amenity_ratio_{cat}"] = 0.0

    rec["amenity_density"] = round(total / TRACT_AREA_KM2, 2)
    records.append(rec)

df_amenities = pd.DataFrame(records)
print(f"\nCompleted: {len(df_amenities)} tracts")
print(f"Mean amenities per tract: {df_amenities['amenity_count_total'].mean():.1f}")
print(f"Tracts with zero amenities: {(df_amenities['amenity_count_total'] == 0).sum()}")

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = "csv/02_amenity_composition.csv"
df_amenities.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_amenities)} rows x {df_amenities.shape[1]} cols)")

# Show count columns summary
count_cols = [c for c in df_amenities.columns if c.startswith("amenity_count_")]
print("\nAmenity count summary:")
print(df_amenities[count_cols].describe().round(1).to_string())